# GeoNet Paddle — Baidu AI Studio / V100

Ce notebook utilise **PaddlePaddle**, pas PyTorch.

Objectif : entraîner la même architecture GeoNet sur le V100 16 Go d'AI Studio.

**Conseil :** exporte d'abord ton `last_multiforme.pt` en `.npz` sur ton PC/Kaggle.
Ainsi Baidu reprend les poids déjà appris ; seul l'état interne de l'optimizer
PyTorch ne peut pas être repris tel quel.


In [ ]:
# 1. Vérifier Paddle + GPU
import paddle

print("Paddle :", paddle.__version__)
print("CUDA compile :", paddle.is_compiled_with_cuda())
print("GPU count :", paddle.device.cuda.device_count())

if not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:
    raise RuntimeError("GPU Paddle indisponible.")

paddle.set_device("gpu:0")
print("Device :", paddle.device.get_device())

!nvidia-smi


In [ ]:
# 2. Chemins
from pathlib import Path

PROJECT_ROOT = Path(
    "/home/aistudio/work/Train_Kaggle_plank_Detector/GeoNet_paddle"
)

# Si ton repo est ailleurs, change uniquement PROJECT_ROOT.
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"GeoNet_paddle introuvable : {PROJECT_ROOT}\n"
        "Clone/upload d'abord ton repo dans /home/aistudio/work."
    )

# Recherche automatique d'un dataset images/ + labels/ sous /home/aistudio/data
DATA_ROOT = Path("/home/aistudio/data")

candidates = []

if DATA_ROOT.is_dir():
    for p in [DATA_ROOT, *DATA_ROOT.rglob("*")]:
        if (
            p.is_dir()
            and (p / "images").is_dir()
            and (p / "labels").is_dir()
        ):
            candidates.append(p)

print("Datasets détectés :")
for i, p in enumerate(candidates):
    print(i, p)

if len(candidates) == 1:
    DATA_DIR = candidates[0]
elif len(candidates) == 0:
    raise FileNotFoundError(
        "Aucun dossier contenant images/ et labels/ sous /home/aistudio/data"
    )
else:
    # Change l'index si nécessaire.
    DATA_DIR = candidates[0]

OUTPUT_DIR = Path(
    "/home/aistudio/work/GeoNet_checkpoints"
)

NPZ_PATH = (
    PROJECT_ROOT
    / "model_poids"
    / "last_multiforme_torch.npz"
)

TRAIN_SCRIPT = (
    PROJECT_ROOT
    / "train"
    / "train_geonet_baidu.py"
)

print("DATA_DIR    :", DATA_DIR)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("TRAIN_SCRIPT:", TRAIN_SCRIPT)
print("NPZ         :", NPZ_PATH)


In [ ]:
# 3. Vérifier rapidement le modèle
import sys
import paddle

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.multiforme_model import build_model

model = build_model(pretrained=False)
model.eval()

x = paddle.randn([1, 3, 256, 256], dtype="float32")

with paddle.no_grad():
    out = model(x)

print("mask_logits   :", tuple(out["mask_logits"].shape))
print("center_logits :", tuple(out["center_logits"].shape))

assert tuple(out["mask_logits"].shape) == (1, 1, 256, 256)
assert tuple(out["center_logits"].shape) == (1, 1, 128, 128)

print("✅ Forward Paddle OK")


In [ ]:
# 4. Configuration V100 16 Go
IMG_SIZE = 640
EPOCHS = 120

BATCH = 4

# Ancien batch effectif Kaggle :
# 4 par GPU * 2 GPU * accumulation 2 = 16
# Ici :
# 4 * 1 GPU * accumulation 4 = 16
ACCUMULATION = 4

WORKERS = 2
VAL_RATIO = 0.15
SEED = 42

LR_HEAD = 1e-5
LR_BACKBONE = 5e-7
WEIGHT_DECAY = 1e-4

FREEZE_BACKBONE_EPOCHS = 2
GRAD_CLIP = 1.0
PATIENCE = 25

USE_AMP = True

print("Batch effectif :", BATCH * ACCUMULATION)


In [ ]:
# 5. Lancer l'entraînement
import subprocess
import sys

cmd = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--data", str(DATA_DIR),
    "--output", str(OUTPUT_DIR),
    "--img-size", str(IMG_SIZE),
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--accumulation", str(ACCUMULATION),
    "--workers", str(WORKERS),
    "--val-ratio", str(VAL_RATIO),
    "--seed", str(SEED),
    "--lr-head", str(LR_HEAD),
    "--lr-backbone", str(LR_BACKBONE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--freeze-backbone-epochs", str(FREEZE_BACKBONE_EPOCHS),
    "--grad-clip", str(GRAD_CLIP),
    "--patience", str(PATIENCE),
]

# Si le NPZ exporté depuis ton checkpoint PyTorch existe, on reprend ses poids.
if NPZ_PATH.is_file():
    cmd += [
        "--init-torch-npz",
        str(NPZ_PATH),
    ]
else:
    print(
        "⚠️ NPZ PyTorch absent : le modèle partira de poids Paddle aléatoires.\n"
        "Je recommande d'exporter ton last_multiforme.pt avant de lancer."
    )

if not USE_AMP:
    cmd.append("--no-amp")

print("Commande :")
print(" ".join(cmd))

result = subprocess.run(cmd)

if result.returncode != 0:
    raise RuntimeError(
        f"Training échoué, code retour = {result.returncode}"
    )


In [ ]:
# 6. Vérifier les checkpoints Paddle
from pathlib import Path
import paddle

for name in [
    "last_multiforme.pdckpt",
    "best_multiforme.pdckpt",
]:
    path = OUTPUT_DIR / name

    if not path.is_file():
        print("❌", path)
        continue

    ckpt = paddle.load(str(path))

    print()
    print("✅", name)
    print("Epoch    :", ckpt.get("epoch"))
    print("Prochaine:", int(ckpt.get("epoch", 0)) + 1)
    print("Best IoU :", ckpt.get("best_iou"))
    print("Taille   :", f"{path.stat().st_size / 1024**2:.2f} MB")


In [ ]:
# 7. Reprise ultérieure depuis le checkpoint Paddle
# Décommente pour reprendre une session Baidu interrompue.

# RESUME = OUTPUT_DIR / "last_multiforme.pdckpt"
#
# cmd_resume = [
#     sys.executable,
#     str(TRAIN_SCRIPT),
#     "--data", str(DATA_DIR),
#     "--output", str(OUTPUT_DIR),
#     "--img-size", str(IMG_SIZE),
#     "--epochs", str(EPOCHS),
#     "--batch", str(BATCH),
#     "--accumulation", str(ACCUMULATION),
#     "--workers", str(WORKERS),
#     "--val-ratio", str(VAL_RATIO),
#     "--seed", str(SEED),
#     "--lr-head", str(LR_HEAD),
#     "--lr-backbone", str(LR_BACKBONE),
#     "--weight-decay", str(WEIGHT_DECAY),
#     "--freeze-backbone-epochs", str(FREEZE_BACKBONE_EPOCHS),
#     "--grad-clip", str(GRAD_CLIP),
#     "--patience", str(PATIENCE),
#     "--resume-paddle", str(RESUME),
# ]
#
# subprocess.run(cmd_resume, check=True)
